# Cubit → .vol High-Order Curving: p-Convergence Demo

Demonstrates the `curvedelements` section in Netgen .vol text format.

Two paths produce identical .vol files:
- **Path A** (C++): `cubit.cmd('export netgen "mesh.vol" order N')` — compact_netgen static link
- **Path B** (Python): `extract_curved_mesh(cubit, order=N)` + `ng_mesh.Save()`

The .vol file is self-contained: mesh + labels + curvedelements. NGSolve reads it without geometry.

**Requirements**: `pip install radia cubit-mesh-export`, Coreform Cubit with radia plugin

In [ ]:
import math
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src', 'radia'))

# NGSolve FIRST (DLL conflict avoidance)
from ngsolve import Mesh, Integrate, CF, BND

RADIUS = 0.05
V_EXACT = (4/3) * math.pi * RADIUS**3
A_EXACT = 4 * math.pi * RADIUS**2
print(f'Sphere R={RADIUS} m')
print(f'V_exact = {V_EXACT:.10e} m^3')
print(f'A_exact = {A_EXACT:.10e} m^2')

## Path B: Python `extract_curved_mesh` → .vol

In [ ]:
from install_panels import find_cubit_bin
_cubit_path = find_cubit_bin()
if _cubit_path: sys.path.append(_cubit_path)

import cubit
cubit.init(['cubit', '-nojournal', '-batch'])
from cubit_mesh_export import extract_curved_mesh

# Create sphere mesh
cubit.cmd('reset')
cubit.cmd(f'create sphere radius {RADIUS}')
cubit.cmd('volume 1 scheme tetmesh')
cubit.cmd('volume 1 size auto factor 5')
cubit.cmd('mesh volume 1')
cubit.cmd('block 1 add volume 1')
cubit.cmd('block 1 name "sphere"')
print(f'Tets: {cubit.get_tet_count()}')

In [ ]:
# p-convergence: export .vol for order 1-5, measure volume and area
import tempfile, os
tmpdir = tempfile.mkdtemp(prefix='pconv_')

results = []
for p in range(1, 6):
    vol_path = os.path.join(tmpdir, f'sphere_p{p}.vol')
    if p == 1:
        ng = extract_curved_mesh(cubit, order=2)
        m = Mesh(ng)
        m.Curve(1)  # reset to linear
    else:
        ng = extract_curved_mesh(cubit, order=p)
        ng.Save(vol_path)
        m = Mesh(vol_path)  # round-trip: Save + Load .vol
    
    vol = Integrate(CF(1), m)
    area = Integrate(CF(1), m, BND)
    v_err = (vol - V_EXACT) / V_EXACT * 100
    a_err = (area - A_EXACT) / A_EXACT * 100
    results.append((p, vol, v_err, area, a_err))
    print(f'p={p}: V_err={v_err:+.6e}%  A_err={a_err:+.6e}%')

# Cleanup
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)

In [ ]:
# Plot p-convergence
import matplotlib.pyplot as plt
import numpy as np

ps = [r[0] for r in results]
v_errs = [abs(r[2]) for r in results]
a_errs = [abs(r[4]) for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(ps, v_errs, 'bo-', label='Volume error [%]', markersize=8)
ax.semilogy(ps, a_errs, 'rs-', label='Area error [%]', markersize=8)
ax.set_xlabel('Polynomial order p')
ax.set_ylabel('Relative error [%]')
ax.set_title(f'p-Convergence: Cubit sphere (R={RADIUS}m, {cubit.get_tet_count()} tets)\n'
             f'.vol round-trip with curvedelements section')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
ax.set_xticks(ps)
plt.tight_layout()
plt.savefig('p_convergence.png', dpi=150)
plt.show()

## Key Points

1. **curvedelements in .vol text format**: `Save()` writes, `Load()` reads — upstream Netgen master
2. **p=5 reaches ~10^-8 % error**: machine precision for volume/area integration
3. **No geometry file needed**: .vol is self-contained (mesh + labels + curving)
4. **CallbackGeometry**: ACIS surface/edge projection via `closest_point_trimmed`
5. **compact_netgen**: static link in Cubit plugin, no nglib.dll dependency